# 팀 프로젝트 - 금융감독기관 보도자료 크롤링
- 금융위원회(FSC): 증권선물위원회 불공정거래 의결 게시판
- 금융감독원(FSS): 보도자료 게시판에서 검색어 "불공정거래"로 필터링한 전체 결과

아래는 실제로 실행해서 결과 CSV까지 만들어낸 최종 코드입니다.

## 금융위원회 보도자료(불공정거래 의결 게시판) 크롤링
- 대상: https://fsc.go.kr/no070200 (목록 페이지 + 각 게시글 상세 페이지)
- 목록 페이지에서 실제 클래스(`.board-list-wrap`, `.subject a`, `.day`, `.info`)를 확인 후 작성했습니다.
- `pages` 인자에 원하는 페이지 번호들을 튜플로 넣으면 여러 페이지를 한 번에 수집합니다. 예: `pages=(1, 2, 3)`

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
BASE_URL = "https://fsc.go.kr"


def get_list(curPage=2):
    """목록 페이지 1개에서 제목/등록일/조회수/상세링크를 수집"""
    url = f"{BASE_URL}/no070200?curPage={curPage}&srchCtgry=&srchEndDt=&srchKey=&srchBeginDt=&srchText="
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    rows = []
    # 게시글 한 건 = .board-list-wrap 안의 li 하나
    for item in soup.select(".board-list-wrap ul li"):
        title_tag = item.select_one(".subject a")
        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)
        detail_url = BASE_URL + title_tag["href"]

        day_tag = item.select_one(".day")
        date = day_tag.get_text(strip=True) if day_tag else ""

        info_tag = item.select_one(".info span")   # "조회수 : 1087" 형태
        views_match = re.search(r"\d+", info_tag.get_text()) if info_tag else None
        views = views_match.group() if views_match else ""

        rows.append({
            "제목": title,
            "등록일": date,
            "조회수": views,
            "링크": detail_url,
        })
    return rows


def get_detail_content(detail_url):
    """상세 페이지에서 본문 텍스트만 추출"""
    res = requests.get(detail_url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    content_tag = soup.select_one(".board-view-wrap .body .cont")
    return content_tag.get_text("\n", strip=True) if content_tag else ""


def crawl_fsc(pages=(2,)):
    """여러 페이지의 목록 + 각 게시글의 본문까지 수집해서 DataFrame으로 반환"""
    all_rows = []
    for page in pages:
        print(f"{page}페이지 목록 수집 중...")
        rows = get_list(page)
        for row in rows:
            print(f"  -> 상세 수집: {row['제목'][:30]}...")
            row["본문"] = get_detail_content(row["링크"])
            all_rows.append(row)
            time.sleep(1)   # 서버 부담을 줄이기 위한 지연

    return pd.DataFrame(all_rows)


df = crawl_fsc(pages=(2,))
df


In [ ]:
# CSV로 저장
file_name = "금융위_보도자료.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"저장 완료: {file_name}")


## 금융감독원(FSS) 보도자료 - 검색어 필터 크롤링 ("불공정거래")
- 대상: https://www.fss.or.kr/fss/bbs/B0000188/list.do?...&searchCnd=1&searchWrd=불공정거래 (전체 394건, 40페이지)
- 검색으로 걸러진 범위라 개수가 크지 않아, 전체 페이지의 본문까지 모두 수집합니다.
- 요청 사이 1초씩 지연을 두기 때문에 약 434번 요청(목록 40 + 상세 394) 기준 7~8분 정도 걸립니다.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
FSS_BASE_URL = "https://www.fss.or.kr"


def fss_search_list(keyword, page_index=1, sdate="", edate=""):
    """검색어로 필터링된 목록 페이지 1개를 가져와서 (행 목록, 전체 페이지 수)를 반환"""
    url = f"{FSS_BASE_URL}/fss/bbs/B0000188/list.do"
    params = {
        "menuNo": "200218", "bbsId": "", "cl1Cd": "",
        "pageIndex": page_index, "sdate": sdate, "edate": edate,
        "searchCnd": 1, "searchWrd": keyword,
    }
    res = requests.get(url, params=params, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    # "페이지 1 / 40" 형태에서 전체 페이지 수를 읽음
    spans = soup.select(".total-count span")
    page_text = spans[1].get_text(" ", strip=True) if len(spans) > 1 else ""
    m = re.search(r"/\s*(\d+)", page_text)
    total_pages = int(m.group(1)) if m else 1

    rows = []
    for tr in soup.select("table.tbl.col.list-data tbody tr"):
        title_tag = tr.select_one("td.title a")
        if not title_tag:
            continue
        tds = tr.find_all("td")
        # td 순서: 0=번호, 1=제목, 2=담당부서, 3=등록일, 4=첨부파일, 5=영상, 6=조회수
        rows.append({
            "번호": tds[0].get_text(strip=True),
            "제목": title_tag.get_text(strip=True),
            "담당부서": tds[2].get_text(strip=True),
            "등록일": tds[3].get_text(strip=True),
            "조회수": tds[-1].get_text(strip=True),
            "링크": FSS_BASE_URL + title_tag["href"],
        })
    return rows, total_pages


def fss_get_detail_content(detail_url):
    """상세 페이지에서 본문 텍스트만 추출"""
    res = requests.get(detail_url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    body_tag = soup.select_one(".krds-bd-view .n-dbdata")
    return body_tag.get_text("\n", strip=True) if body_tag else ""


def fss_search_crawl(keyword, sdate="", edate=""):
    """검색어에 해당하는 게시글 전체(모든 페이지)의 목록 + 본문을 수집"""
    all_rows = []
    page = 1
    total_pages = 1
    while page <= total_pages:
        print(f"{page}페이지 목록 수집 중...")
        rows, total_pages = fss_search_list(keyword, page, sdate, edate)
        if page == 1:
            print(f"검색어 '{keyword}' 전체 {total_pages}페이지")
        for row in rows:
            print(f"  -> 상세 수집: {row['제목'][:30]}...")
            row["본문"] = fss_get_detail_content(row["링크"])
            all_rows.append(row)
            time.sleep(1)   # 서버 부담을 줄이기 위한 지연
        page += 1

    return pd.DataFrame(all_rows)


# 검색어 "불공정거래"로 필터링된 게시글 전체(394건, 40페이지)의 본문까지 수집
fss_search_df = fss_search_crawl("불공정거래")
fss_search_df


In [ ]:
# CSV로 저장
file_name = "금감원_보도자료_불공정거래.csv"
fss_search_df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"저장 완료: {file_name} (총 {len(fss_search_df)}건)")


### 실행 결과
위 검색어 필터 크롤링은 2026-09-23에 실제로 실행했고, 아래 파일로 저장을 완료했습니다.
- 파일: `금감원_보도자료_불공정거래.csv`
- 총 394건 (40페이지 전체)
- 컬럼: 번호, 제목, 담당부서, 등록일, 조회수, 링크, 본문

다시 실행하면 최신 데이터로 새로 수집되며, 394건 기준 약 7~8분 걸립니다.